# Informations

# 📑 Payments Transactional Report

## 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/263225346/BRD+-+Payments+Transactional+Report)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-532)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/DBTC-958)**  
  A summary of findings from UAT.

# Authentication and basic libraries

In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback
import os

from pyspark.sql.functions import col, when, lit, month, year, concat, left, expr, trim, substring, concat_ws, coalesce, length, max

from typing import Any
 
from delta import DeltaTable  # type: ignore

from typing import Optional
 
import delta  # type: ignore
from pyspark.sql import SparkSession
 
from io import BytesIO
import pandas as pd
import json
import sys
 
from datetime import datetime
from pytz import timezone

"""
Accessing the Hive metastore requires specific considerations:
 
1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.
 
2. To access both Unity catalog and AWS services, an access token from Amazon is required.
 
3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html
 
4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.
 
5. After you authenticate successfully querying the Hive metastore is also working.
"""
 
try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared"
 
# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
   
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    
 
secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)
 
aws_key = aws_keys[0] if aws_keys else None
 
# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
 
if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                          
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')
 
    S3_client = boto3.client('s3', region_name='eu-central-1')                          
 
 
# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']
 
# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

## Input Parameters
 
**report_args:** it is a json which contains all the appropriate parameters to run the report
 
`
{ "portfolio_name": "Mercury", "output_file_prefix": "Mercury_Payments_report",  "report_output": "/Volumes/qa-cdh-delivery/internalreports/Mercury/Payments/" }
 
 
**run_date:** is the date value to run the report. The run_date is empty when it is called by the orchestrator and that means "today"

In [0]:
athens_tz = timezone('Europe/Athens')
 
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")
 
try:    
    report_name = dbutils.widgets.get("report_name")
except Exception as e:
    raise ValueError("Missing required input: report_name")
print(f"Report {report_name} is started")
 
try:
    run_date = dbutils.widgets.get("run_date")
    if run_date is None or run_date == "": 
        raise ValueError("ERROR: run_date has no value. Remove the run_date completely if you want to run the job using the current date or add a value in the following format 'YYYY-MM-DD'")    
    run_date_str = run_date.replace('-', '')
    run_date = datetime.strptime(run_date, '%Y-%m-%d').date()
except Exception as e:
    print(f"exception {e}")
    run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    run_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
 
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

#Set the RefenceDate to be the previous working date in referal to run_Date 

# Step 1: Define the DF with the previous business day based on run_date
refdf = spark.sql(f"""
    SELECT PreviousBusinessDay 
    FROM `{env_prefix}-cdh`.reference.ref_calendar 
    WHERE CalendarDate = '{run_date}'
                """)  
# Extract the value into a Python variable
referenceDate = refdf.collect()[0]["PreviousBusinessDay"] if refdf.count() > 0 else None
print("Referenced Date:", referenceDate)
### End of RefenceDate setup 
 
# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "portfolio_name" not in json_data: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

portfolioName = json_data['portfolio_name']
output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']

print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")
 
# print(f"now is: {now_datetime_str}")

# Create Temp Views

In [0]:
dexposure = spark.sql(f"""
                    WITH all_exposures as (
                      SELECT CDI, AgreementKey, HasActiveSettlementFlag, AssetSegment, issecuredflag, SubServicer, ExposureKey, Bucket, casecode,BusinessDate,PortfolioName,
                      ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY BusinessDate DESC) as rn
                      FROM `{env_prefix}-cdh`.derivative.dexposure
                      WHERE PortfolioName LIKE ('%{portfolioName}%')
                    )
                    SELECT DISTINCT * FROM all_exposures WHERE rn = 1
                    """)
dexposure.createOrReplaceTempView("dexposure")

dexposure.createOrReplaceTempView("dexposure")
dparty = spark.sql(f"""
                    SELECT DISTINCT CDI, Segment, TaxRegistrationNumber, FirstNameGR , LastNameGR , FirstNameENG, LastNameENG, GroupCode, GroupName, HighBalanceFlag
                    FROM `{env_prefix}-cdh`.derivative.dparty
                    WHERE sourcesystem = 'profits' 
                    AND BusinessDate = '{referenceDate}' 
                    AND PortfolioName LIKE ('%{portfolioName}%')
                   """)
dparty.createOrReplaceTempView("dparty")

dcollection = spark.sql(f"""
                       SELECT dc.*
                       FROM `{env_prefix}-cdh`.derivativeactive.dcollection dc
                       join `{env_prefix}-cdh`.reference.ref_calendar calendar on calendar.CalendarDate = dc.actualdate
                       left join dexposure de on 1=1
                          and (dc.exposurekey=de.exposurekey 
                              or 
                              (dc.exposurekey=substring(de.exposurekey,4,100) and dc.portfolioName = de.portfolioname)
                          )
                       WHERE dc.BusinessDate = '{referenceDate}' 
                       AND dc.PortfolioName LIKE ('%{portfolioName}%')
                       AND DATEDIFF(dc.BusinessDate,dc.Actualdate)<=365
                         """)
dcollection.createOrReplaceTempView("dcollection")

settlementexposure = spark.sql(f"""
                              SELECT ExposureKey,BusinessDate,ismigratedflag
                              FROM `{env_prefix}-cdh`.primary.settlementexposure 
                              WHERE BusinessDate = '{referenceDate}' 
                              AND PortfolioName LIKE ('%{portfolioName}%')
                               """)
settlementexposure.createOrReplaceTempView("settlementexposure")

dunderlaw = spark.sql(f"""
                     select CDI, InLawFlag from 
                     (
                      SELECT DISTINCT CDI, InLawFlag ,row_number() over(partition by CDI order by case when sourcesystem = 'process' then 0 else 1 end desc) as rn
                      FROM `{env_prefix}-cdh`.derivative.dunderlaw
                      WHERE sourcesystem IN ('qca', 'process')
                      AND  UnderLawType='N3869'
                      AND BusinessDate = 
                        ( SELECT max(BusinessDate) AS BusinessDate
                          FROM `{env_prefix}-cdh`.derivative.dunderlaw
                          WHERE sourcesystem IN ('qca', 'process')
                        )
                      )x where x.rn=1     
                     """) #Selecting the maximum BusinessDate from underlaw for customer snapshot as descibred in DOMUS-75 By Pavlos
dunderlaw.createOrReplaceTempView("dunderlaw")
#defines secured perimeter for monza
servicerreportsecuredperimeter = spark.sql(f"""
                                WITH latest_servicer_report_securedperimeter as (
                                  SELECT max(ReferenceDate) as MaxReferenceDate
                                  FROM `{env_prefix}-reportinputfiles`.investor.servicer_report_securedperimeter
                                  WHERE ReferenceDate <= '{referenceDate}'
                              )
                                      SELECT
                                          DebtorCDI,
                                          CASE 
                                              WHEN SecUnsec = 'Unsecured D' THEN 'No'
                                              WHEN SecUnsec = 'Secured D' THEN 'Yes'
                                          END AS IsSecuredFlag,
                                          ReferenceDate
                                      FROM `{env_prefix}-reportinputfiles`.investor.servicer_report_securedperimeter sec_per
                                      JOIN latest_servicer_report_securedperimeter lsr 
                                          ON sec_per.ReferenceDate = lsr.MaxReferenceDate
                                      WHERE PortfolioName ILIKE '%monza%'
""")
servicerreportsecuredperimeter.createOrReplaceTempView("servicerreportsecuredperimeter")


kymasubperimeter = spark.sql(f"""  -- EK0701
    SELECT *
    FROM `{env_prefix}-cdh`.reference.kyma_subperimeter_split
""")
kymasubperimeter.createOrReplaceTempView("kymasubperimeter")


# Code for selects

In [0]:
secured_perimeter_portfolios =['MONZA PORTFOLIO - D', 'MONZA PORTFOLIO - RETAIL', 'MONZA PORTFOLIO - WHOLESALE']

In [0]:
# Build join conditions outside for clarity and reuse
join_dexposure = [
    col("de.ExposureKey") == col("dc.ExposureKey")
]
join_dparty = col("dp.CDI") == coalesce(col("dc.PayerCDI"), col("dc.PrimaryOwnerCDI"))

join_settlementexposure = [
    col("se.ExposureKey") == col("dc.ExposureKey"),
    col("se.BusinessDate") == col("dc.actualdate")
]
join_dunderlaw = col("ul.cdi") == col("dp.CDI")
join_servicerreportsecuredperimeter = (col("srp.DebtorCDI") == col("dc.PrimaryOwnerCDI"))
join_kymasubperimeter = expr("right(de.AgreementKey, 8)") == col("`Contract_ID`")   #EK0701

Internal = (
    dcollection.alias("dc")
    .join(dexposure.alias("de"), join_dexposure, "left")
    .join(dparty.alias("dp"), join_dparty, "left")
    .join(settlementexposure.alias("se"), join_settlementexposure, "left")
    .join(dunderlaw.alias("ul"), join_dunderlaw, "left")
    .join(servicerreportsecuredperimeter.alias("srp"), join_servicerreportsecuredperimeter, "left")
    .join(kymasubperimeter.alias("k"), join_kymasubperimeter, "left")    #EK0701
    .withColumn(
        "Tranche",
        when(col("dc.PortfolioName").like('%KYMA%'), col("k.`Sub_Perimeter_Formated`"))   #EK0701
        .when(col("dp.Segment").like('%-B%'), 'Tranche B').otherwise('Tranche A')
    )
    .withColumn(
        "Payment Amount EUR",
        expr(f"`{env_prefix}-cdh`.derivative.converttoeur(dc.currency, dc.amount, dc.recorddate)").cast("double")
    )
    .select(
        col("dc.DCollectionKey").alias("DCollectionKey"),
        col("dc.PortfolioName").alias("Portfolio Name"),
        col("dc.CollectionSourceKey").alias("Transaction unique id"),
        col("dp.TaxRegistrationNumber").alias("Tax Number"),
        coalesce(col("dc.PayerCDI"), col("dc.PrimaryOwnerCDI")).alias("CIF Number"),
        col("dc.PrimaryOwnerCDI").alias("CIF Number Primary Debtor"),
        col("de.AgreementKey").alias("AgreementNumber"),
        col("dc.ExposureKey").alias("Account Number"),
        col("dc.SourceExposureKey").alias("Host Account Number"),
        col("dc.actualdate").alias("Payment Date"),
        col("dc.recorddate").alias("Record Date"),
        col("Payment Amount EUR"),
        when(col("dc.IsPaymentFlag") == True, "Yes")
        .when(col("dc.IsPaymentFlag") == False, "No")
        .alias("Is Payment Flag"),
        when(
            col("dc.PaymentType") == "Είσπραξη από πελάτη, κατόπιν ρύθμισης που έγινε στη Cepal.",
            "Είσπραξη από πελάτη"
        ).otherwise(col("dc.PaymentType")).alias("PaymentType"),
        col("dc.PaymentTypeEng").alias("PaymentTypeEng"),
        col("dc.currency").alias("Currency"),
        when(col("dc.FinalReason") == "Matched", "Yes").otherwise("No").alias("Is Payment Matched"),
        col("dc.MatchedDate").alias("Matched date"),
        trim(concat_ws(" ", coalesce(col("dp.FirstNameGR"), lit("")), coalesce(col("dp.LastNameGR"), lit("")))).alias("Party Name"),
        trim(concat_ws(" ", coalesce(col("dp.FirstNameENG"), lit("")), coalesce(col("dp.LastNameENG"), lit("")))).alias("PartyNameEng"),
        col("dp.GroupCode").alias("GroupID"),
        col("dp.GroupName").alias('GroupName'),
        col("dp.segment").alias("Customer Segment"),
        when(
            (col("de.HasActiveSettlementFlag") == True) & (col("se.ismigratedflag") == True), "Yes"
        ).otherwise("No").alias("Settlement from Migration flag"),
        when(col("ul.InLawFlag") == 1, "Yes")
        .when(col("ul.InLawFlag") == 0, "No")
        .alias("L3869 Court Approval flag"),
        when(col("de.HasActiveSettlementFlag") == True, "Yes")
        .when(col("de.HasActiveSettlementFlag") == False, "No")
        .alias("Payment within Active Settl"),
        col("de.AssetSegment").alias("Asset Segment"),
        col("dc.AllocationExternalCompany").alias("External Company Name"),
        coalesce(col("dc.AllocationAgent"),lit("Unassigned")).alias("Agent Name"),
        when(col("dc.AllocationExternalCompany").isNotNull() & (col("dc.AllocationExternalCompany") != "Unassigned"), lit("Other"))
        .otherwise(
            coalesce(col("dc.AllocationTeam"),lit("Unassigned"))
            ).alias("Team Name"),
        when(col("dc.AllocationExternalCompany").isNotNull() & (col("dc.AllocationExternalCompany") != "Unassigned"), lit("External LF"))
        .otherwise(
            coalesce(col("dc.AllocationChannel"),lit("Unassigned"))
            ).alias("Channel Name"),
        col("dc.AssignmentProcess").alias("Assignment Process"),
        col("dc.ProcessType").alias("Packet Process Type"),
        when(col("dc.PortfolioName").isin(secured_perimeter_portfolios), col("srp.IsSecuredFlag")) #update secured perimeter
            .otherwise(when(col("de.IsSecuredFlag") == True, "Yes").when(col("de.IsSecuredFlag") == False, "No"))
            .alias("Is Secured Flag"),
        col("de.SubServicer").alias("SubServicer"),
        col("Tranche"),
        coalesce(col("dp.HighBalanceFlag").cast("string"), lit("")).alias("High Balance Flag"),
        col("de.Bucket"),
        col("dc.DiasNumber"),
        col("dc.DiasChannel"),
        col("dc.PaymentTypeGroup").alias("Payment Type Group")

    )
    .distinct()
)

# Nullify columns for not matched records in a single pass
columns_to_keep = [
    "DCollectionKey", "Collection Key", "Portfolio Name", "Transaction unique id",
    "CIF Number", "CIF Number Primary Debtor", "Account Number", "Host Account Number",
    "Payment Date", "Record Date", "Is Payment Flag", "PaymentType","PaymentTypeEng", "Currency",
    "Agent Name", "Team Name", "Channel Name", "External Company Name", "Packet Process Type", "Is Payment Matched", 
    "Matched date", "Payment Amount EUR", "Bucket", "DiasNumber", "DiasChannel", "Payment Type Group"

]

exprs = [
    col(c) if c in columns_to_keep else when(col("Is Payment Matched") == "No", lit(None)).otherwise(col(c)).alias(c)
    for c in Internal.columns
]

Internal = Internal.select(*exprs)
Internal.createOrReplaceTempView("Internal")

# Internal

In [0]:
columns = [
    "DCollectionKey", "Portfolio Name", "Transaction unique id", "Tax Number", "CIF Number", "CIF Number Primary Debtor", "AgreementNumber",
    "Account Number", "Host Account Number", "Payment Date", "Record Date", "Payment Amount EUR", "Is Payment Flag",
    "PaymentType","PaymentTypeEng", "Currency", "Is Payment Matched", "Matched date", "Party Name", "PartyNameEng",
    "GroupID", "GroupName", "Customer Segment", "Settlement from Migration flag",
    "L3869 Court Approval flag", "Payment within Active Settl", "Asset Segment", "Agent Name",
    "Team Name", "Channel Name", "External Company Name", "Assignment Process", "Packet Process Type", "Is Secured Flag",
    "SubServicer", "TRANCHE", "High Balance Flag", "Bucket", "DiasNumber", "DiasChannel", "Payment Type Group"
]

Final_internal = Internal.select(*columns).fillna("")
Final_internal.createOrReplaceTempView("Final_internal")

In [0]:
#Define the output directory path
output_dir = f"{report_output}{run_date_str}/"

# Define the output file path
output_path = f"{output_dir}{output_file_prefix}_Internal_{referenceDate}.xlsx"
output_path_pivot = f"{output_dir}{output_file_prefix}_Internal_aggregated_{referenceDate}.xlsx"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

display("output_dir : ", output_dir)
display("output_path : ", output_path)
display("output_path_pivot : ", output_path_pivot)
display("output_file_prefix : ", output_file_prefix)


In [0]:
#spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
output_df = Final_internal.toPandas()

In [0]:
import numpy as np

# assuming final_internal is already a Pandas DataFrame
df = output_df.copy()

# filter once, with vectorized boolean masks
mask = (
    (df["Is Payment Matched"] == "Yes") &
    (df["PaymentTypeEng"].str.contains("recurrin", case=False, na=False))
)

filtered = df.loc[mask].copy()

# extract Year and Month once
filtered["Payment Date"] = pd.to_datetime(filtered["Payment Date"], errors="coerce")
filtered["Year"] = pd.to_datetime(filtered["Payment Date"], errors="coerce").dt.year
filtered["Month"] = filtered["Payment Date"].dt.month

# build AssignmentProcessForLF with efficient numpy where logic
filtered["AssignmentProcessForLF"] = np.where(
    filtered["External Company Name"].notna() & (filtered["External Company Name"] != ""),
    np.where(
        filtered["Assignment Process"].str.contains("offer", case=False, na=False),
        "Offer",
        "Collection",
    ),
    "Other",
)

# group + aggregate efficiently
pivot = (
    filtered.groupby([
        "Portfolio Name",
        "Year",
        "Month",
        "External Company Name",
        "Channel Name",
        "Team Name",
        "AssignmentProcessForLF",
        "Agent Name",
        "Packet Process Type"
    ], dropna=False, as_index=False)
    .agg(MonthlyPayments=("Payment Amount EUR", "sum"))
)

# ensure type is double (float64)
pivot["MonthlyPayments"] = pivot["MonthlyPayments"].astype("float64")

# final sort
pivot = pivot.sort_values(
    ["Portfolio Name", "Year", "Month", "External Company Name", "Channel Name", "Team Name"]
)


In [0]:
output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    pivot.to_excel(writer, index=False)

with open(output_path_pivot, "wb") as f:
    f.write(output_buffer.getvalue())
 
print(f"Results successfully written to Volume at: {output_path_pivot}")


output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    output_df.to_excel(writer, index=False)

with open(output_path, "wb") as f:
    f.write(output_buffer.getvalue())
 
print(f"Results successfully written to Volume at: {output_path}")